# Apex Retail Intelligence — Raw & Landing Zone Script

**Author:** Aashi Phulera
**Programme:** Celebal Technologies | CEI'26 Internship Programme — Major Project
**Layer:** Raw Ingestion + Landing Zone
**Technology:** PySpark, Databricks, Unity Catalog Volumes

---

### Purpose
This notebook handles the first two phases of the Medallion pipeline:
- **Phase 1 — Raw Zone:** Ingests all incoming CSVs (customer, product, sales — historical and incremental), casts every column to String format, and organizes output into separate `raw/historical/` and `raw/incremental/` directories.
- **Phase 2 — Landing Zone:** Converts Raw CSVs to Parquet format, then dynamically validates row counts against the corresponding `audit_landing*.csv` files, producing a structured PASS/FAIL report. The pipeline halts on any audit failure.

### Prerequisites
Run the setup cell below first to ensure the catalog, schemas, and volume exist. This script is safe to re-run — all writes use `overwrite` mode, so re-running does not create duplicates.

### Independently Executable
This notebook can be run standalone from a fresh cluster — it reads directly from the `apex_retail.bronze.incoming_data` volume and requires no state from other notebooks.

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS apex_retail")
spark.sql("CREATE SCHEMA IF NOT EXISTS apex_retail.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS apex_retail.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS apex_retail.GOLD_tables")
spark.sql("CREATE VOLUME IF NOT EXISTS apex_retail.bronze.incoming_data")

DataFrame[]

In [0]:
display(dbutils.fs.ls("/Volumes/apex_retail/bronze/incoming_data"))

path,name,size,modificationTime
dbfs:/Volumes/apex_retail/bronze/incoming_data/customer_historical.csv,customer_historical.csv,82586,1785815717000
dbfs:/Volumes/apex_retail/bronze/incoming_data/customer_historical_audit.csv,customer_historical_audit.csv,48,1785815716000
dbfs:/Volumes/apex_retail/bronze/incoming_data/customer_incremental.csv,customer_incremental.csv,106981,1785815717000
dbfs:/Volumes/apex_retail/bronze/incoming_data/customer_incrementalaudit.csv,customer_incrementalaudit.csv,49,1785815716000
dbfs:/Volumes/apex_retail/bronze/incoming_data/customer_incrementalaudit_silver.csv,customer_incrementalaudit_silver.csv,41,1785815716000
dbfs:/Volumes/apex_retail/bronze/incoming_data/customer_silver_audit.csv,customer_silver_audit.csv,48,1785815716000
dbfs:/Volumes/apex_retail/bronze/incoming_data/landing/,landing/,0,1786222804739
dbfs:/Volumes/apex_retail/bronze/incoming_data/product_historical.csv,product_historical.csv,128050,1785815717000
dbfs:/Volumes/apex_retail/bronze/incoming_data/product_historical_audit.csv,product_historical_audit.csv,47,1785815716000
dbfs:/Volumes/apex_retail/bronze/incoming_data/product_incremental.csv,product_incremental.csv,139293,1785815717000


In [0]:
# ============================================
# PHASE 1: RAW ZONE INGESTION
# Reads all incoming CSVs, casts every column to string,
# and organizes output into separate historical/incremental raw directories.
# ============================================

base_in = "/Volumes/apex_retail/bronze/incoming_data"
base_raw = "/Volumes/apex_retail/bronze/incoming_data/raw"   # <-- fixed: lives inside the volume

def ingest_raw(file_name, dataset_name, load_type):
    """Reads a CSV, forces all columns to string, writes to raw/ zone."""
    df = spark.read.option("header", True).csv(f"{base_in}/{file_name}")
    
    for c in df.columns:
        df = df.withColumn(c, df[c].cast("string"))
    
    out_path = f"{base_raw}/{load_type}/{dataset_name}"
    df.write.mode("overwrite").option("header", True).csv(out_path)
    
    row_count = df.count()
    print(f"[RAW] {dataset_name} ({load_type}): {row_count} rows -> {out_path}")
    return row_count

# Historical loads
ingest_raw("customer_historical.csv", "customer", "historical")
ingest_raw("product_historical.csv", "product", "historical")
ingest_raw("sales_historical.csv", "sales", "historical")

# Incremental loads
ingest_raw("customer_incremental.csv", "customer", "incremental")
ingest_raw("product_incremental.csv", "product", "incremental")
ingest_raw("sales_incremental.csv", "sales", "incremental")

print("\n✅ Phase 1 Raw ingestion complete.")

[RAW] customer (historical): 1052 rows -> /Volumes/apex_retail/bronze/incoming_data/raw/historical/customer
[RAW] product (historical): 1043 rows -> /Volumes/apex_retail/bronze/incoming_data/raw/historical/product
[RAW] sales (historical): 1002 rows -> /Volumes/apex_retail/bronze/incoming_data/raw/historical/sales
[RAW] customer (incremental): 1053 rows -> /Volumes/apex_retail/bronze/incoming_data/raw/incremental/customer
[RAW] product (incremental): 1041 rows -> /Volumes/apex_retail/bronze/incoming_data/raw/incremental/product
[RAW] sales (incremental): 1000 rows -> /Volumes/apex_retail/bronze/incoming_data/raw/incremental/sales

✅ Phase 1 Raw ingestion complete.


In [0]:
# Debug: check actual column names in the audit files
sample_audit = spark.read.option("header", True).csv(
    "/Volumes/apex_retail/bronze/incoming_data/customer_historical_audit.csv"
)
sample_audit.printSchema()
display(sample_audit)

root
 |-- table_name: string (nullable = true)
 |-- row_count: string (nullable = true)



table_name,row_count
customer_historical,1052


In [0]:
# ============================================
# PHASE 2: LANDING ZONE + AUDIT VALIDATION (robust version)
# ============================================

base_audit = "/Volumes/apex_retail/bronze/incoming_data"
base_landing = "/Volumes/apex_retail/bronze/incoming_data/landing"

all_files = [f.name for f in dbutils.fs.ls(base_audit) if f.name.endswith(".csv")]

def find_audit_file(dataset_name, load_type):
    """Dynamically locate the matching audit_landing*.csv file, excluding silver audits."""
    candidates = [
        f for f in all_files
        if dataset_name in f.lower()
        and load_type in f.lower()
        and "silver" not in f.lower()
        and "audit" in f.lower()
    ]
    if not candidates:
        raise FileNotFoundError(f"No landing audit file found for {dataset_name} ({load_type}). Checked against: {all_files}")
    if len(candidates) > 1:
        print(f"⚠️ Multiple matches for {dataset_name} ({load_type}): {candidates} — using {candidates[0]}")
    return candidates[0]

def get_expected_count(audit_df, dataset_name):
    """Safely extract row_count, filtering by table_name if multiple rows exist."""
    rows = audit_df.collect()
    if len(rows) == 0:
        raise ValueError(f"Audit file for {dataset_name} is empty!")
    
    # If multiple rows, try to match table_name; else just take first row
    if len(rows) > 1:
        matched = [r for r in rows if dataset_name in r["table_name"].lower()]
        row = matched[0] if matched else rows[0]
    else:
        row = rows[0]
    
    raw_value = row["row_count"]
    if raw_value is None or str(raw_value).strip() == "":
        raise ValueError(f"row_count is empty for {dataset_name}. Row content: {row}")
    
    return int(str(raw_value).strip())

audit_report = []

def convert_and_validate(dataset_name, load_type):
    raw_path = f"/Volumes/apex_retail/bronze/incoming_data/raw/{load_type}/{dataset_name}"
    df = spark.read.option("header", True).csv(raw_path)
    actual_count = df.count()

    out_path = f"{base_landing}/{load_type}/{dataset_name}"
    df.write.mode("overwrite").parquet(out_path)

    audit_file = find_audit_file(dataset_name, load_type)
    print(f"   -> matched audit file: {audit_file}")  # debug line
    audit_df = spark.read.option("header", True).csv(f"{base_audit}/{audit_file}")
    expected_count = get_expected_count(audit_df, dataset_name)

    status = "PASS" if actual_count == expected_count else "FAIL"
    audit_report.append({
        "dataset": dataset_name,
        "load_type": load_type,
        "audit_file": audit_file,
        "expected": expected_count,
        "actual": actual_count,
        "status": status
    })

    if status == "FAIL":
        raise Exception(f"AUDIT FAILED for {dataset_name} ({load_type}): expected {expected_count}, got {actual_count}")

    print(f"[LANDING] {dataset_name} ({load_type}): {actual_count} rows -> {out_path} | {status}")

for name in ["customer", "product", "sales"]:
    for load_type in ["historical", "incremental"]:
        print(f"\nProcessing {name} ({load_type})...")
        convert_and_validate(name, load_type)

print("\n📋 Structured Audit Report:")
report_df = spark.createDataFrame(audit_report)
display(report_df)

print("\n✅ Phase 2 Landing zone + audit validation complete.")


Processing customer (historical)...
   -> matched audit file: customer_historical_audit.csv
[LANDING] customer (historical): 1052 rows -> /Volumes/apex_retail/bronze/incoming_data/landing/historical/customer | PASS

Processing customer (incremental)...
   -> matched audit file: customer_incrementalaudit.csv
[LANDING] customer (incremental): 1053 rows -> /Volumes/apex_retail/bronze/incoming_data/landing/incremental/customer | PASS

Processing product (historical)...
   -> matched audit file: product_historical_audit.csv
[LANDING] product (historical): 1043 rows -> /Volumes/apex_retail/bronze/incoming_data/landing/historical/product | PASS

Processing product (incremental)...
   -> matched audit file: product_incrementalaudit.csv
[LANDING] product (incremental): 1041 rows -> /Volumes/apex_retail/bronze/incoming_data/landing/incremental/product | PASS

Processing sales (historical)...
   -> matched audit file: sales_historical_audit.csv
[LANDING] sales (historical): 1002 rows -> /Volumes

actual,audit_file,dataset,expected,load_type,status
1052,customer_historical_audit.csv,customer,1052,historical,PASS
1053,customer_incrementalaudit.csv,customer,1053,incremental,PASS
1043,product_historical_audit.csv,product,1043,historical,PASS
1041,product_incrementalaudit.csv,product,1041,incremental,PASS
1002,sales_historical_audit.csv,sales,1002,historical,PASS
1000,sales_incrementalaudit.csv,sales,1000,incremental,PASS



✅ Phase 2 Landing zone + audit validation complete.
